# Steam Trajectory — Data Exploration

Loading the fronkongames/steam-games-dataset **JSON** export (switched from CSV after validation revealed row-level misalignment — see notes in `kaggle_loader.py`). This notebook confirms the JSON loads cleanly, then runs cohort selection.

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("/Users/pmacias/Dropbox/steamproject")
print(os.getcwd())

/Users/pmacias/Dropbox/steamproject


In [2]:
from steam_trajectory.ingest.kaggle_loader import KaggleLoader

# NOTE: adjust filename below to match whatever the JSON actually
# downloaded as (likely games.json — confirm in data/raw/ first)
loader = KaggleLoader("data/raw/games.json")
df = loader.df
print(df.shape)
df.head()

(135043, 9)


,AppID,Name,Release date,Developers,Publishers,Price,total_reviews,review_score_percent,Genres
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",NaN,NaN,0.00,0,NaN,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",minori,MangaGamer,5.24,255,98.823529,Adventure
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",Somer Games,8floor,4.99,24,87.500000,Casual
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",유진게임즈,유진게임즈,8.99,0,NaN,"Casual, Indie, Simulation"
4,3631080,Maze Quest VR,"Apr 24, 2025",Reality Expanded LLC,Reality Expanded LLC,4.99,0,NaN,"Action, Early Access"


## Validation
Confirming the JSON loaded cleanly before trusting `select_cohort()`'s output. No CSV-style alignment issues are possible here (JSON fields are explicitly keyed, not positional), but worth still checking for data-quality issues (missing dates, bad AppIDs, etc.) independent of the format itself.

In [3]:
import pandas as pd

# 1. Do release dates parse cleanly?
release_dates = pd.to_datetime(df["Release date"], errors="coerce")
print("Unparseable release dates:", release_dates.isna().sum(), "of", len(df))

Unparseable release dates: 0 of 135043


In [4]:
# 2. Is AppID unique and always an integer? Our whole schema's
# primary key depends on this being true.
print("Duplicate AppIDs:", df["AppID"].duplicated().sum())
print("AppID dtype:", df["AppID"].dtype)

Duplicate AppIDs: 0
AppID dtype: int64


In [5]:
# 3. Is Price always numeric and non-negative?
print("Price dtype:", df["Price"].dtype)
print("Negative prices:", (df["Price"] < 0).sum())
print("Missing prices:", df["Price"].isna().sum())

Price dtype: float64
Negative prices: 0
Missing prices: 0


In [6]:
# 4. Spot-check: do name/date/price/genre all look like they
# plausibly belong to the same game, across a random sample?
df.sample(10, random_state=1)[["Name", "Release date", "Price", "Genres"]]

,Name,Release date,Price,Genres
109723,Blacksmith Simulator,"Nov 28, 2025",5.99,"Adventure, Casual, Indie, Simulation"
26780,"Ethyrial, Echoes of Yore Playtest","Jan 24, 2025",0.00,NaN
78923,ETERNAL EXILE: BENEATH THE DARKNESS,"Aug 20, 2025",15.99,"Action, Indie"
51423,Where is my car? Playtest,"Sep 30, 2025",0.00,NaN
125316,PengPong: Prologue,"Feb 20, 2026",0.00,"Action, Casual, Indie, Free To Play"
88912,Stagehand Survival Simulator Playtest,"Feb 13, 2023",0.00,NaN
90366,Smash and Bash Monsters: Prologue,"Jul 22, 2024",0.00,"Action, Casual, Indie, RPG, Free To Play"
127770,Chronicles: Immortal Sect,"Mar 28, 2026",0.00,"Adventure, Massively Multiplayer, RPG, Free To..."
45862,Video World,"Jan 21, 2021",0.00,Indie
33603,Prince Of Wallachia,"Apr 28, 2021",0.59,"Action, Adventure, Indie"


In [7]:
# 5. total_reviews / review_score_percent sanity check
print(df[["total_reviews", "review_score_percent"]].describe())

       total_reviews  review_score_percent
count   1.350430e+05          82979.000000
mean    1.102677e+03             75.824456
std     3.110025e+04             23.868394
min     0.000000e+00              0.000000
25%     0.000000e+00             64.964021
50%     4.000000e+00             81.818182
75%     3.900000e+01             94.444444
max     8.815087e+06            100.000000


## Cohort selection
If everything above looks clean, select the actual research cohort: 2019–2022 releases with 500+ reviews.

In [8]:
cohort = loader.select_cohort(
    release_start="2019-01-01",
    release_end="2022-12-31",
    min_reviews=500,
    sample_size=200,
)
print(cohort.shape)
cohort[["AppID", "Name", "Release date", "total_reviews"]].head(10)

(200, 9)


,AppID,Name,Release date,total_reviews
0,1030210,Torchlight III,"Oct 13, 2020",11230
1,852090,Penko Park,"Oct 23, 2020",578
2,1100420,Praetorians - HD Remaster,"Jan 24, 2020",985
3,1248130,Farming Simulator 22,"Nov 21, 2021",84512
4,1808780,祖玛少女/Zuma Girls,"May 1, 2022",684
5,1100990,Aimbeast,"May 13, 2020",1978
6,1734320,Brutal Orchestra,"Dec 17, 2021",1800
7,1367590,Tormented Souls,"Aug 26, 2021",6089
8,1218250,We Went Back,"Apr 3, 2020",4627
9,1043390,FlowScape,"Aug 15, 2019",1313


## Time to scrape Steamcharts 

In [9]:
from steam_trajectory.ingest.steamcharts_scraper import SteamChartsScraper

scraper = SteamChartsScraper()
test_appids = cohort["AppID"].head(3).tolist()
print("Testing:", test_appids)

for appid in test_appids:
    history = scraper.get_monthly_history(appid)
    print(f"\nAppID {appid}: {len(history)} months of data")
    print(history[:3])  # first 3 months, just to eyeball


Testing: [1030210, 852090, 1100420]

AppID 1030210: 73 months of data
[{'appid': 1030210, 'month': 'June 2026', 'avg_players': 22.82, 'peak_players': 45, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}, {'appid': 1030210, 'month': 'May 2026', 'avg_players': 21.42, 'peak_players': 50, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}, {'appid': 1030210, 'month': 'April 2026', 'avg_players': 18.74, 'peak_players': 38, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}]

AppID 852090: 31 months of data
[{'appid': 852090, 'month': 'July 2025', 'avg_players': 1.96, 'peak_players': 11, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}, {'appid': 852090, 'month': 'December 2024', 'avg_players': 1.64, 'peak_players': 7, 'est_owners_low': None, 'est_owners_high': None, 'source': 'steamcharts'}, {'appid': 852090, 'month': 'November 2024', 'avg_players': 2.25, 'peak_players': 10, 'est_owners_low': Non

In [10]:
from steam_trajectory.ingest.steamcharts_scraper import SteamChartsScraper

scraper = SteamChartsScraper()
all_appids = cohort["AppID"].tolist()

all_histories, failures = scraper.get_monthly_history_batch(all_appids)

print(f"\nDone. {len(all_histories)} monthly records, {len(failures)} games failed.")
print(failures)

Processed 20/200 games (6 failures so far)...
Processed 40/200 games (13 failures so far)...
Processed 60/200 games (17 failures so far)...
Processed 80/200 games (22 failures so far)...
Processed 100/200 games (26 failures so far)...
Processed 120/200 games (32 failures so far)...
Processed 140/200 games (37 failures so far)...
Processed 160/200 games (42 failures so far)...
Processed 180/200 games (47 failures so far)...
Processed 200/200 games (51 failures so far)...

Done. 9656 monthly records, 51 games failed.
[{'appid': 1808780, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1808780'}, {'appid': 1043390, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1043390'}, {'appid': 1345740, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1345740'}, {'appid': 1477790, 'error': '500 Server Error: Internal Server Error for url: https://steamcharts.com/app/1477790'}, {'appid': 

In [11]:
# Pull extra candidates up front (30% buffer, tune if needed) so
# that after failures are dropped, you still land near 200 usable games.
larger_cohort = loader.select_cohort(
    release_start="2019-01-01",
    release_end="2022-12-31",
    min_reviews=500,
    sample_size=260,
)

all_appids = larger_cohort["AppID"].tolist()
all_histories, failures = scraper.get_monthly_history_batch(all_appids)

failed_appids = {f["appid"] for f in failures}
final_cohort = larger_cohort[~larger_cohort["AppID"].isin(failed_appids)].reset_index(drop=True)

print(f"Final cohort size: {len(final_cohort)}")

Processed 20/260 games (4 failures so far)...
Processed 40/260 games (11 failures so far)...
Processed 60/260 games (15 failures so far)...
Processed 80/260 games (19 failures so far)...
Processed 100/260 games (22 failures so far)...
Processed 120/260 games (26 failures so far)...
Processed 140/260 games (31 failures so far)...
Processed 160/260 games (36 failures so far)...
Processed 180/260 games (41 failures so far)...
Processed 200/260 games (45 failures so far)...
Processed 220/260 games (54 failures so far)...
Processed 240/260 games (60 failures so far)...
Processed 260/260 games (65 failures so far)...
Final cohort size: 195


In [12]:
final_appids = set(final_cohort["AppID"])
final_histories = [record for record in all_histories if record["appid"] in final_appids]
print(f"{len(final_histories)} monthly records for the final {len(final_cohort)}-game cohort")

12605 monthly records for the final 195-game cohort


In [13]:
from steam_trajectory.db.connection import get_connection
from steam_trajectory.db.writer import DatabaseWriter

conn = get_connection("steam_project.db")
writer = DatabaseWriter(conn)

# 1. Games — iter_games yields dicts whose keys already match
# insert_game's parameter names exactly, so **unpacking just works
for game in loader.iter_games(final_cohort):
    writer.insert_game(**game)

# 2. Genre links — get_or_create_genre handles the lookup-table
# bookkeeping automatically inside link_game_genre
for appid, genre_name in loader.iter_genres(final_cohort):
    writer.link_game_genre(appid, genre_name)

# 3. Monthly metrics from the scrape
for record in final_histories:
    writer.insert_monthly_metric(**record)

writer.commit()
print("Database load complete.")

Database load complete.


In [14]:
import pandas as pd

print("games:", pd.read_sql("SELECT COUNT(*) as n FROM games", conn)["n"][0])
print("game_genres:", pd.read_sql("SELECT COUNT(*) as n FROM game_genres", conn)["n"][0])
print("monthly_metrics:", pd.read_sql("SELECT COUNT(*) as n FROM monthly_metrics", conn)["n"][0])

# A real join, exercising exactly the multi-table SQL this whole
# schema was designed to support
pd.read_sql("""
    SELECT g.name, g.release_date, gen.name AS genre, COUNT(m.month) AS months_tracked
    FROM games g
    JOIN game_genres gg ON g.appid = gg.appid
    JOIN genres gen ON gg.genre_id = gen.genre_id
    JOIN monthly_metrics m ON g.appid = m.appid
    GROUP BY g.appid, gen.name
    LIMIT 10
""", conn)

games: 195
game_genres: 533
monthly_metrics: 12605


,name,release_date,genre,months_tracked
0,GearCity,"Jan 14, 2022",Indie,145
1,GearCity,"Jan 14, 2022",Simulation,145
2,GearCity,"Jan 14, 2022",Strategy,145
3,ShellShock Live,"May 22, 2020",Action,136
4,ShellShock Live,"May 22, 2020",Casual,136
5,ShellShock Live,"May 22, 2020",Indie,136
6,ShellShock Live,"May 22, 2020",Massively Multiplayer,136
7,ShellShock Live,"May 22, 2020",Strategy,136
8,Tales from the Borderlands,"Feb 16, 2021",Adventure,139
9,Infernax,"Feb 14, 2022",Action,53


## GearCity shows up as 145 months tracked, which doesn't reach back to 2022. This is because of early release. Let's look to see how common this is

In [15]:
pd.read_sql("""
    SELECT g.name, g.release_date, MIN(m.month) AS earliest_tracked_month
    FROM games g
    JOIN monthly_metrics m ON g.appid = m.appid
    GROUP BY g.appid
""", conn)

,name,release_date,earliest_tracked_month
0,GearCity,"Jan 14, 2022",April 2015
1,ShellShock Live,"May 22, 2020",April 2015
2,Tales from the Borderlands,"Feb 16, 2021",April 2015
3,Infernax,"Feb 14, 2022",April 2022
4,Hurtworld,"Dec 10, 2019",April 2016
...,...,...,...
190,Mage and Monsters,"Oct 12, 2022",April 2023
191,Rush Rally 3,"Nov 24, 2022",April 2023
192,Richman 11,"Oct 19, 2022",April 2023
193,Aonatsu Line,"Dec 2, 2022",April 2023
